# Steel Surface Crack Detection using Transfer Learning and Explainable AI
**Course:** AI Project EST  
**Project Objective:** Implementation of an automated system for steel plate inspection using Convolutional Neural Networks, specifically targeting micro-level surface defects.

---

## 1. Import Libraries
Utilization of TensorFlow for deep learning, Scikit-Learn for performance metrics, and Seaborn for statistical visualization.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import cv2

# Check for hardware acceleration
print("Number of GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

## 2. Dataset Loading and Preprocessing
The dataset is processed using rescaling and data augmentation techniques to enhance model generalization in industrial environments.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_PATH = 'dataset/'

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True
)

val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

print("Class Indices:", train_data.class_indices)

## 3. Sample Visualization
Visual inspection of the dataset labels and image quality.

In [ ]:
def show_samples(data_gen):
    images, labels = next(data_gen)
    plt.figure(figsize=(10, 10))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i])
        label = "Crack" if labels[i] == 0 else "No Crack"
        plt.title(f"Label: {label}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(train_data)

## 4. Model Architecture
Implementation of Transfer Learning using a pre-trained MobileNetV2 backbone. A custom classification head is appended for the binary classification task.

In [ ]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # Freeze base model for initial training

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## 5. Training Methodology

### Phase 1: Classifier Head Training
Training the custom dense layers while keeping the feature extraction layers frozen.

In [ ]:
print("Commencing Phase 1 training...")
history_phase1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

### Phase 2: Model Fine-Tuning
The base model is unfrozen and retrained with a lower learning rate to optimize the feature extraction layers for this specific domain.

In [ ]:
print("Commencing Phase 2 (Fine-tuning)...")
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5), 
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=3
)
model.save("steel_crack_model.keras")
print("Model saved to steel_crack_model.keras for deployment.")

## 6. Performance Evaluation
Quantitative analysis using Classification Reports and Confusion Matrices.

In [ ]:
val_data.reset()
y_true = val_data.classes
y_pred_prob = model.predict(val_data)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=list(train_data.class_indices.keys())))

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=train_data.class_indices.keys(), 
            yticklabels=train_data.class_indices.keys())
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix')
plt.show()

## 7. Explainable AI: Grad-CAM Visualization
Interpretability through Class Activation Maps (Grad-CAM) to identify feature importance in model decision making.

In [ ]:
def get_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        class_channel = preds[:, 0]
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    heatmap = get_gradcam_heatmap(img_array, model, 'out_relu')
    heatmap = np.uint8(255 * heatmap)
    jet = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    jet_heatmap = tf.keras.utils.array_to_img(jet_heatmap).resize((224, 224))
    jet_heatmap = tf.keras.utils.img_to_array(jet_heatmap)

    superimposed_img = jet_heatmap * 0.4 + (img_array[0] * 255)
    superimposed_img = tf.keras.utils.array_to_img(superimposed_img)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1); plt.imshow(img); plt.title("Original Sample"); plt.axis("off")
    plt.subplot(1, 2, 2); plt.imshow(superimposed_img); plt.title("Grad-CAM Attribution Map"); plt.axis("off")
    plt.show()

try:
    sample_path = os.path.join(DATASET_PATH, 'crack', os.listdir(os.path.join(DATASET_PATH, 'crack'))[0])
    display_gradcam(sample_path)
except: pass

## 8. Predictive Analytics
Functionality for inference on novel image samples.

In [ ]:
def predict_image(path):
    img = tf.keras.utils.load_img(path, target_size=(224, 224))
    x = tf.keras.utils.img_to_array(img) / 255.0
    x = np.expand_dims(x, axis=0)
    pred = model.predict(x)[0][0]
    label = "No Crack" if pred > 0.5 else "Crack"
    confidence = pred if pred > 0.5 else 1 - pred
    print(f"Prediction Result: {label} (Confidence: {confidence*100:.2f}%)")
    plt.imshow(img); plt.axis('off'); plt.show()

## 9. Conclusion
The implemented system demonstrates a robust pipeline for automated surface inspection. Future developments will focus on integrating more diverse defect categories and optimizing for real-time inference latency.

## 10. Real-Time High-Resolution Inspection (Sliding Window)
This section simulates a production environment. To detect micro-level cracks in high-resolution images without shrinking them and losing quality, we slide a 224x224 inspection window across the large image. If a crack is found, a bounding box is drawn.

In [ ]:
def inspect_high_res_image(image_path, model, window_size=(224, 224), step_size=112):
    # Load high-resolution image using OpenCV
    original_img = cv2.imread(image_path)
    if original_img is None:
        print(f"Error loading image: {image_path}")
        return
    
    # Convert BGR to RGB for correct display and prediction
    original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    
    # Create a copy to draw bounding boxes on
    output_img = original_img.copy()
    
    h, w, _ = original_img.shape
    crack_found = False
    
    print(f"Scanning high-resolution image of size {w}x{h}...")
    
    # Sliding window loop
    for y in range(0, h - window_size[1] + 1, step_size):
        for x in range(0, w - window_size[0] + 1, step_size):
            # Extract the 224x224 patch
            patch = original_img[y:y+window_size[1], x:x+window_size[0]]
            
            # Preprocess the patch for the model
            patch_array = tf.keras.utils.img_to_array(patch) / 255.0
            patch_array = np.expand_dims(patch_array, axis=0)
            
            # Predict
            pred = model.predict(patch_array, verbose=0)[0][0]
            
            # If Crack is detected (pred < 0.5)
            if pred < 0.5:
                crack_found = True
                # Draw a red rectangle (bounding box) on the output image
                cv2.rectangle(output_img, (x, y), (x + window_size[0], y + window_size[1]), (255, 0, 0), 3)
                # Add text label
                cv2.putText(output_img, f"Crack: {1-pred:.2f}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
                
    # Display the final result
    plt.figure(figsize=(12, 12))
    plt.imshow(output_img)
    plt.title("Production Simulation: High-Res Inspection Result")
    plt.axis("off")
    plt.show()
    
    if crack_found:
        print("ALARM: Defect detected on the steel plate!")
    else:
        print("PASS: No defects detected.")

# Example Usage:
try:
    print("Testing Sliding Window Algorithm on a sample...")
    inspect_high_res_image(sample_path, model, step_size=50)
except Exception as e:
    print("Please provide a valid high-resolution image path.", e)